# 06 — Synthetic Division

**Topic 6 of the math journey.** Last time the **Factor Theorem** told us: $(x-c)$ is a
factor of $p(x)$ exactly when $p(c)=0$. But to *use* it we need to actually **divide**
$p(x)$ by $(x-c)$ and read the remainder. **Synthetic division** is a fast, compact way to
do exactly that — no $x$'s to carry, just numbers.

Mental picture: ordinary long division of numbers ($437 \div 7$) has a cousin for
polynomials. Synthetic division is the *speed version* of that cousin, but only for divisors
of the special shape $x-c$.

**Why it matters.** Synthetic division lets us (1) divide quickly, (2) **evaluate** a
polynomial fast (this is *Horner's method*, used in real numerical software and inside AI
libraries), and (3) **test and peel off roots** to fully factor higher-degree polynomials.

## Quick recall (from factoring)
- **Factor Theorem:** $(x-c)$ divides $p(x)$ $\iff p(c)=0$.
- **Remainder Theorem:** dividing $p(x)$ by $(x-c)$ leaves remainder $p(c)$.
- Difference of cubes: $x^3-27=(x-3)(x^2+3x+9)$.

## What we cover today
1. Polynomial **long division** (the full method first).
2. The **Division Algorithm** for polynomials (theorem + proof).
3. **Synthetic division**: the shortcut for divisor $x-c$, and *why* it works.
4. Synthetic division as **fast evaluation** (Horner's method) — link to the Remainder Theorem.
5. **Testing roots** and the **Rational Root Theorem** to fully factor.


## 1. Polynomial long division

Just like number long division, we divide the highest term, multiply back, subtract, and
bring down the next term — repeat.

**Example:** divide $p(x)=2x^3 - 3x^2 + 0x - 4$ by $d(x)=x-2$.

```
            2x^2 +  x  + 2
          ___________________________
  x - 2 ) 2x^3 - 3x^2 + 0x - 4
          2x^3 - 4x^2                 <- 2x^2 * (x-2)
          -----------
                 x^2 + 0x             <- subtract, bring down
                 x^2 - 2x             <-  x  * (x-2)
                 ---------
                       2x - 4         <- subtract, bring down
                       2x - 4         <-  2  * (x-2)
                       ------
                            0         <- remainder
```

So $2x^3-3x^2-4 = (x-2)(2x^2+x+2) + 0$. The **quotient** is $2x^2+x+2$ and the
**remainder** is $0$ — meaning $x=2$ is a root (Factor Theorem ✓).

General form of any division:
$$ p(x) = d(x)\,q(x) + r(x), \qquad \deg r < \deg d. $$


In [1]:
import sympy as sp                       # sympy = exact symbolic algebra
x = sp.symbols('x')

p = 2*x**3 - 3*x**2 - 4
d = x - 2
q, r = sp.div(p, d, x)                     # sp.div returns (quotient, remainder)
print("quotient :", q)                     # 2*x**2 + x + 2
print("remainder:", r)                     # 0
print("check p = d*q + r:", sp.expand(d*q + r))   # rebuilds the original p


quotient : 2*x**2 + x + 2
remainder: 0
check p = d*q + r: 2*x**3 - 3*x**2 - 4


## 2. The Division Algorithm for polynomials (theorem + proof)

**Theorem.** Let $p(x)$ and $d(x)$ be polynomials with $d(x)\neq 0$. Then there exist
**unique** polynomials $q(x)$ (quotient) and $r(x)$ (remainder) such that
$$ p(x) = d(x)\,q(x) + r(x), \qquad \text{with } r(x)=0 \text{ or } \deg r < \deg d. $$

**Proof — existence (by reducing the degree).**
If $p=0$ or $\deg p < \deg d$, take $q=0$ and $r=p$; done. Otherwise let
$\deg p = n$, $\deg d = m$ with $n\ge m$, and let $a$ and $b$ be the leading coefficients of
$p$ and $d$. Form
$$ p_1(x) = p(x) - \frac{a}{b}x^{\,n-m}\,d(x). $$
The term $\frac{a}{b}x^{n-m}d(x)$ has the same leading term $a x^n$ as $p$, so that term
**cancels** and $\deg p_1 < n$. We have strictly lowered the degree. Repeat on $p_1$, then
$p_2$, and so on. Because the degree drops each step, after finitely many steps the leftover
has degree $< m$; call it $r$. Adding up all the pieces $\frac{a}{b}x^{n-m}+\cdots$ gives
$q$, and $p = d\,q + r$. ∎

**Proof — uniqueness.**
Suppose $p = dq_1+r_1 = dq_2+r_2$ with both remainders below $\deg d$. Subtract:
$d(q_1-q_2) = r_2-r_1$. The right side has degree $< m$. If $q_1\neq q_2$, the left side has
degree $\ge m$ (a non-zero multiple of $d$) — contradiction. So $q_1=q_2$, and then
$r_1=r_2$. ∎

This theorem is the **foundation** under both long division and synthetic division.


## 3. Synthetic division — the shortcut for divisor $x-c$

When the divisor is exactly $x-c$, all the $x$'s in long division are predictable
bookkeeping. Synthetic division keeps **only the coefficients**.

**Procedure** to divide $p(x)=a_n x^n + \dots + a_1 x + a_0$ by $(x-c)$:
1. Write the coefficients $a_n, a_{n-1}, \dots, a_0$ in a row (use $0$ for missing powers).
2. Write $c$ on the left.
3. Bring the first coefficient straight down.
4. Multiply it by $c$, write the result under the next coefficient, and **add**.
5. Repeat: multiply the new number by $c$, add to the next coefficient.
6. The last number is the **remainder**; the others are the coefficients of the **quotient**
   (degree one less than $p$).

**Example:** $2x^3-3x^2+0x-4$ divided by $x-2$, so $c=2$.

```
  c=2 |  2   -3    0   -4
      |       4    2    4
      +--------------------
         2    1    2  |  0
        (quotient coeffs)  (remainder)
```
Quotient $=2x^2+x+2$, remainder $=0$ — same answer as the long division above, much faster.

**Why it works.** In long division by $x-c$, at each step you multiply the current quotient
term by $x-c$ and subtract. Subtracting $(\text{term})\cdot(-c)$ is the same as **adding**
$(\text{term})\cdot c$ to the next coefficient — which is exactly the "multiply by $c$ and
add" rule. Synthetic division just records those additions and drops the $x$-powers (they
only mark position). So it is genuine long division with the redundant symbols removed. ∎


In [2]:
def synthetic_division(coeffs, c):
    '''Divide a polynomial (given by coeffs, highest power first) by (x - c).
    Returns (quotient_coeffs, remainder).  Pure Python — no library needed.'''
    result = [coeffs[0]]                    # step 3: bring down the first coefficient
    for a in coeffs[1:]:
        result.append(a + result[-1]*c)     # step 4-5: multiply by c, then add
    *quotient, remainder = result           # last entry is the remainder
    return quotient, remainder

# Example: 2x^3 - 3x^2 + 0x - 4  divided by (x - 2):
q, r = synthetic_division([2, -3, 0, -4], 2)
print("quotient coeffs:", q, "  remainder:", r)   # [2, 1, 2]  remainder 0

# Cross-check with sympy:
import sympy as sp
x = sp.symbols('x')
print("sympy says:", sp.div(2*x**3 - 3*x**2 - 4, x - 2, x))


quotient coeffs: [2, 1, 2]   remainder: 0
sympy says: (2*x**2 + x + 2, 0)


## 4. Synthetic division = fast evaluation (Horner's method)

The remainder from dividing $p(x)$ by $(x-c)$ **is $p(c)$** (Remainder Theorem). So the last
number in synthetic division gives you $p(c)$ — a quick way to **evaluate** a polynomial.

Look at what the "multiply by $c$, add" loop actually computes for
$p(x)=a_3x^3+a_2x^2+a_1x+a_0$:
$$ p(c) = ((a_3\cdot c + a_2)\cdot c + a_1)\cdot c + a_0. $$
This nested form is **Horner's method**. It uses only $n$ multiplications and $n$ additions —
the cheapest way to evaluate a polynomial, and what fast numerical libraries actually use.

**Worked example:** evaluate $p(x)=2x^3-3x^2-4$ at $x=2$ using the loop: start $2$;
$2\cdot2-3=1$; $1\cdot2+0=2$; $2\cdot2-4=0$. So $p(2)=0$. ✓


In [3]:
import numpy as np

def horner_eval(coeffs, c):
    '''Evaluate a polynomial at x=c using Horner's method (the synthetic-division loop).'''
    total = 0
    for a in coeffs:                        # left to right: nested multiply-add
        total = total*c + a
    return total

coeffs = [2, -3, 0, -4]
print("horner p(2) =", horner_eval(coeffs, 2))        # 0  (= remainder above)
print("numpy  p(2) =", np.polyval(coeffs, 2))         # np.polyval also uses Horner internally
print("numpy  p(3) =", np.polyval(coeffs, 3))         # 2*27 - 3*9 - 4 = 23


horner p(2) = 0
numpy  p(2) = 0
numpy  p(3) = 23


## 5. Finding and peeling off roots — the Rational Root Theorem

To **fully factor** a polynomial of degree $\ge 3$, we hunt for one root $c$, divide out
$(x-c)$ with synthetic division, and repeat on the smaller quotient. But which numbers
should we test?

**Rational Root Theorem.** Let $p(x)=a_n x^n+\dots+a_0$ have **integer** coefficients. If a
reduced fraction $\tfrac{s}{t}$ is a root, then $s$ divides $a_0$ (the constant term) and $t$
divides $a_n$ (the leading coefficient).

**Proof.** Suppose $p(s/t)=0$ with $\gcd(s,t)=1$. Then
$$ a_n\frac{s^n}{t^n}+a_{n-1}\frac{s^{n-1}}{t^{n-1}}+\dots+a_0=0. $$
Multiply by $t^n$:
$$ a_n s^n + a_{n-1}s^{n-1}t + \dots + a_1 s\,t^{n-1} + a_0 t^n = 0. $$
Every term except $a_n s^n$ has a factor of $t$, so $t \mid a_n s^n$; since $\gcd(s,t)=1$,
$t\mid a_n$. Symmetrically, every term except $a_0 t^n$ has a factor of $s$, so $s\mid a_0$. ∎

**Strategy (full factorisation):**
1. List candidate roots $\pm\frac{\text{divisors of }a_0}{\text{divisors of }a_n}$.
2. Test each with synthetic division (remainder $0$ = root).
3. Divide out the factor $(x-c)$; repeat on the quotient.
4. Stop when the quotient is quadratic (or smaller) — finish with factoring from lesson 5.

**Worked example:** $p(x)=x^3-6x^2+11x-6$. Candidates are $\pm1,\pm2,\pm3,\pm6$. Test $c=1$:
synthetic gives remainder $0$, quotient $x^2-5x+6=(x-2)(x-3)$. So
$$ p(x)=(x-1)(x-2)(x-3). $$


In [4]:
import sympy as sp
x = sp.symbols('x')

p = x**3 - 6*x**2 + 11*x - 6

# Test each rational-root candidate by hand with our synthetic_division function:
def synthetic_division(coeffs, c):
    result = [coeffs[0]]
    for a in coeffs[1:]:
        result.append(a + result[-1]*c)
    *quotient, remainder = result
    return quotient, remainder

coeffs = [1, -6, 11, -6]
for c in [1, 2, 3, 6, -1, -2, -3, -6]:
    q, r = synthetic_division(coeffs, c)
    print(f"  c={c:>2}: remainder={r:>3}  {'<-- ROOT' if r == 0 else ''}")

print("full factorisation:", sp.factor(p))       # (x - 1)(x - 2)(x - 3)
print("roots:", sp.solve(p, x))                  # [1, 2, 3]


  c= 1: remainder=  0  <-- ROOT
  c= 2: remainder=  0  <-- ROOT
  c= 3: remainder=  0  <-- ROOT
  c= 6: remainder= 60  
  c=-1: remainder=-24  
  c=-2: remainder=-60  
  c=-3: remainder=-120  
  c=-6: remainder=-504  
full factorisation: (x - 3)*(x - 2)*(x - 1)
roots: [1, 2, 3]


## Summary — what you should now be able to do

1. Perform **polynomial long division** and write $p = d\,q + r$ with $\deg r < \deg d$.
2. State and **prove the Division Algorithm** (existence + uniqueness).
3. Carry out **synthetic division** to divide by $x-c$ quickly, and explain *why* it works.
4. Use synthetic division / **Horner's method** to **evaluate** a polynomial fast, connecting
   the last number to the **Remainder Theorem** ($p(c)$).
5. Use the **Rational Root Theorem** (proved) to list candidate roots, then peel off factors
   with repeated synthetic division to **fully factor** a higher-degree polynomial.


## Exercises (20)

Do each **by hand first** (write out the synthetic-division row), then check with
`synthetic_division(...)` or `sp.div(...)`. The last block asks for proofs.

**A. Long division**
1. Divide $x^2 + 5x + 6$ by $x + 2$.
2. Divide $x^3 - 1$ by $x - 1$.
3. Divide $2x^3 + 3x^2 - 1$ by $x + 1$ (mind the missing $x$ term → use $0$).
4. Divide $x^3 + x^2 - 4x - 4$ by $x^2 - 4$ (divisor is *not* $x-c$ — use long division).
5. Write the result of #3 in the form $p = d\,q + r$.

**B. Synthetic division**
6. Divide $x^3 - 2x^2 - 5x + 6$ by $x - 1$.
7. Divide $2x^3 + x^2 - 13x + 6$ by $x - 2$.
8. Divide $x^4 - 16$ by $x - 2$ (fill missing powers with $0$).
9. Divide $3x^3 - 4x + 5$ by $x + 2$ (so $c=-2$).
10. From #6, is $x=1$ a root? Explain using the remainder.

**C. Evaluation & remainder**
11. Use Horner's method (the synthetic loop) to find $p(3)$ for $p(x)=x^3-2x^2+4$.
12. Find $p(-1)$ for $p(x)=2x^4 - x^2 + 3$ by synthetic division.
13. Without dividing fully, what is the remainder when $x^5 - 1$ is divided by $x-1$? Why?
14. For $p(x)=x^3+kx-10$, find $k$ so that $x-2$ is a factor. (Hint: $p(2)=0$.)
15. Evaluate $p(x)=x^3-6x^2+11x-6$ at $x=2$ and explain what the answer tells you.

**D. Full factoring & proofs (raise the bar)**
16. List all Rational-Root-Theorem candidates for $x^3 + 2x^2 - 5x - 6$, find one root, and
    fully factor it.
17. Fully factor $x^3 - 7x + 6$ using synthetic division.
18. Prove the **Remainder Theorem**: dividing $p(x)$ by $(x-c)$ leaves remainder $p(c)$.
19. Prove the **Rational Root Theorem** for the special case of a *monic* cubic
    $x^3+a_2x^2+a_1x+a_0$ with integer coefficients: every rational root is an integer
    dividing $a_0$.
20. Explain, using the Division Algorithm, why the remainder when dividing by a *degree-1*
    polynomial $x-c$ must be a constant.


In [5]:
# Helper you may reuse for the exercises:
def synthetic_division(coeffs, c):
    result = [coeffs[0]]
    for a in coeffs[1:]:
        result.append(a + result[-1]*c)
    *quotient, remainder = result
    return quotient, remainder

# Your work for exercises 1-5 (long division). Use sp.div(p, d, x) to check.


In [6]:
# Your work for exercises 6-10 (synthetic division).


In [7]:
# Your work for exercises 11-15 (evaluation & remainder).


In [8]:
# Your work for exercises 16-20 (full factoring & proofs). Write proofs in a markdown cell.


## How to run this notebook with `uv`

From the project folder `C:\dev\math`:

```powershell
uv sync                 # install dependencies (first time / after changes)
uv run jupyter lab      # open JupyterLab, then open notebooks/06-synthetic-division.ipynb
```

No new libraries this time (`sympy` + `numpy`, both already installed). Add any future one
with `uv add <name>`.
